# SPY Overnight Dip Buy - Performance Charts

Replicates the dip strategy (prev close > 200d SMA AND prev close < 5d EMA AND down close; buy at close, exit next open) from daily bars and renders:
1. Cumulative returns: strategy frictionless, strategy net of fees, SPY buy & hold
2. Rolling 252-day Sharpe: same three curves
3. Drawdown comparison: strategy vs SPY buy & hold
4. Monthly returns heatmap (net of fees)

Fees modeled as $5.685 per round trip (backtest total $5,092 / 896 round trips).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from QuantConnect import DataNormalizationMode, Resolution

qb = QuantBook()
spy = qb.add_equity('SPY', Resolution.DAILY).symbol

# split-adjusted prices for the signal, total-return prices for performance
px = qb.history(spy, datetime(2009,1,1), datetime(2026,7,17), Resolution.DAILY,
                data_normalization_mode=DataNormalizationMode.SPLIT_ADJUSTED)
tr = qb.history(spy, datetime(2009,1,1), datetime(2026,7,17), Resolution.DAILY,
                data_normalization_mode=DataNormalizationMode.TOTAL_RETURN)
px = px.loc[spy]
tr = tr.loc[spy]
print('bars:', len(px), len(tr), '| range:', px.index[0], '->', px.index[-1])

In [ ]:
# dip signal on split-adjusted prices, decided at close of day t using data through t
close = px['close']
sma200 = close.rolling(200).mean()
ema5 = close.ewm(span=5, adjust=False).mean()
dip = (close > sma200) & (close < ema5) & (close < close.shift(1))

# overnight return within the total-return series: buy at close t, sell at open t+1
close_tr = tr['close']
open_tr = tr['open']
night_ret = open_tr.shift(-1) / close_tr - 1

df = pd.DataFrame({'close': close, 'dip': dip, 'night': night_ret})
df = df.loc['2010-01-01':'2026-07-10']
# a night trade on day t earns overnight into t+1; drop the last row (no next open)
df = df.iloc[:-1]
trades = df[df['dip'] & df['night'].notna()]
print('signal days:', len(df), '| dip nights:', int(df['dip'].sum()), '| tradable:', len(trades))
print('win rate: %.2f%% | mean: %.4f%%' % (100*(trades['night']>0).mean(), 100*trades['night'].mean()))

In [ ]:
# daily strategy return series: night return on dip nights, flat otherwise
FEE = 5.685  # dollars per round trip (backtest: $5,092 total / 896 round trips)
eq_fric = eq_net = 100000.0
fric_eq, net_eq = [], []
for t, row in df.iterrows():
    r = row['night'] if (bool(row['dip']) and pd.notna(row['night'])) else 0.0
    eq_fric *= (1.0 + r)
    eq_net = eq_net * (1.0 + r) - (FEE if r != 0.0 else 0.0)
    fric_eq.append(eq_fric)
    net_eq.append(eq_net)

strat_fric = pd.Series(fric_eq, index=df.index)
strat_net = pd.Series(net_eq, index=df.index)
spy_bh = tr['close'].reindex(df.index)
spy_bh = spy_bh / spy_bh.iloc[0] * 100000.0
print('frictionless: %.2f%% | net fees: %.2f%% | SPY BH: %.2f%%' % (
    100*(strat_fric.iloc[-1]/100000-1), 100*(strat_net.iloc[-1]/100000-1), 100*(spy_bh.iloc[-1]/100000-1)))


In [ ]:
# CHART 1: cumulative returns
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(strat_fric.index, strat_fric/1e3, label='Strategy overnight dip (frictionless)', lw=1.6)
ax.plot(strat_net.index, strat_net/1e3, label='Strategy overnight dip (net of fees)', lw=1.6)
ax.plot(spy_bh.index, spy_bh/1e3, label='SPY buy & hold', lw=1.4, alpha=0.8)
ax.set_yscale('log')
ax.set_ylabel('Equity ($k, log scale)')
ax.set_title('SPY Overnight Dip Buy vs SPY Buy & Hold - Cumulative Returns (2010-2026)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# CHART 2: rolling 252-day Sharpe - strategy with and without fees vs SPY buy & hold
r_fric = strat_fric.pct_change().fillna(0.0)
r_net = strat_net.pct_change().fillna(0.0)
r_spy = spy_bh.pct_change().fillna(0.0)
def roll_sharpe(r, w=252):
    m = r.rolling(w, min_periods=w).mean()
    s = r.rolling(w, min_periods=w).std()
    return (m/s)*np.sqrt(252)
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(roll_sharpe(r_fric), label='Strategy (frictionless)', lw=1.3)
ax.plot(roll_sharpe(r_net), label='Strategy (net of fees)', lw=1.3)
ax.plot(roll_sharpe(r_spy), label='SPY buy & hold', lw=1.3, alpha=0.8)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Rolling 252d Sharpe')
ax.set_title('Rolling 252-Day Sharpe Ratio')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# full-sample Sharpes for reference
print('full-sample daily Sharpe - frictionless: %.2f | net: %.2f | SPY BH: %.2f' % (
    r_fric.mean()/r_fric.std()*np.sqrt(252),
    r_net.mean()/r_net.std()*np.sqrt(252),
    r_spy.mean()/r_spy.std()*np.sqrt(252)))
print('note: per-trade Sharpe (1.60-1.86) is over trade nights only; the daily-curve Sharpe scales by sqrt(21.7%% exposure) to ~0.75, so SPY can exceed it on a calendar-day basis in strong bull runs.')


In [ ]:
# CHART 3: drawdown comparison (net of fees vs SPY only)
def dd(eq): return eq/eq.cummax() - 1.0
fig, ax = plt.subplots(figsize=(12,6))
ax.fill_between(strat_net.index, dd(strat_net)*100, 0, label='Strategy (net of fees)', alpha=0.5)
ax.fill_between(spy_bh.index, dd(spy_bh)*100, 0, label='SPY buy & hold', alpha=0.4)
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown: Strategy vs SPY Buy & Hold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('maxDD net: %.2f%% | SPY BH: %.2f%%' % (100*dd(strat_net).min(), 100*dd(spy_bh).min()))


In [ ]:
# CHART 4: monthly returns heatmap (net of fees)
monthly = (1 + r_net).resample('ME').prod() - 1.0
hm = pd.DataFrame({'y': monthly.index.year, 'm': monthly.index.month, 'r': monthly.values})
piv = hm.pivot(index='y', columns='m', values='r')
piv.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig, ax = plt.subplots(figsize=(11,7))
im = ax.imshow(piv.values*100, cmap='RdYlGn', vmin=-6, vmax=6, aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
for i in range(len(piv)):
    for j in range(12):
        v = piv.values[i,j]
        if pd.notna(v):
            ax.text(j, i, '%.1f' % (v*100), ha='center', va='center', fontsize=7)
ax.set_title('Monthly Returns (%) - Strategy Net of Fees')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# reconciliation vs backtest log
print('RECONCILIATION vs backtest 468b93cbf6e8016de66a5ff9c64f909b')
print('dip nights: research %d vs backtest 896' % int(df['dip'].sum()))
print('frictionless cum: research %.2f%% vs backtest fill-log +88.05%%' % (100*(strat_fric.iloc[-1]/1e5-1)))
print('net cum: research %.2f%% vs backtest +79.18%%' % (100*(strat_net.iloc[-1]/1e5-1)))
print('NOTE: research uses total-return prices, so ex-dividend nights are slightly richer than the backtest fill log - small positive gap expected (same effect seen on the gate study).')